In [26]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np

In [27]:
FILE = '../data/data_cleaned.csv'

"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)

print(df.columns.tolist())

['decision_datetime', 'is.randomized', 'snooze.status', 'intransit', 'avail', 'send', 'returned.message', 'response', 'dec.precipitation.chance', 'uid', 'decision_idx_nogap', 'decision_date', 'decision_slot', 'steps30', 'steps30pre', 'steps10', 'weather', 'temp', 'loc']


In [28]:
df = df[df['avail'] == True]
df['date'] = pd.to_datetime(df["decision_date"])
df['weekday'] = df['date'].dt.weekday
df['slot'] = df['decision_slot'].astype(int)
# df['study_day'] = (df['date'] - df.groupby('uid')['date'].transform('min')).dt.days + 1
df['study_day'] = df.groupby('uid')['date'].rank(method='dense').astype(int)

# df = df.sort_values(["uid", "decision_datetime"]).reset_index(drop=True)
# dosages = []
# for _, g in df.groupby("uid", sort=False):
#     d, prev_a = 0.0, 0
#     out = []
#     for a in g["send"].values:
#         d = 0.95 * d + (1 if prev_a > 0 else 0)
#         out.append(d)
#         prev_a = a
#     dosages.extend(out)
# df["dosage"] = dosages


df = df.sort_values(["uid", "decision_datetime"]).reset_index(drop=True)
dosages = []
for _, g in df.groupby("uid", sort=False):
    d, prev_a, prev_day = 0.0, 0, None
    out = []
    for a, day in zip(g["send"].values, g["study_day"].values):
        # gap 检测: study_day 跳了 (不是 +0 也不是 +1) -> reset
        if prev_day is not None and day - prev_day > 1:
            d, prev_a = 0.0, 0
        d = 0.95 * d + (1 if prev_a > 0 else 0)
        out.append(d)
        prev_a = a
        prev_day = day
    dosages.extend(out)
df["dosage"] = dosages

df["reward"] = np.log(df["steps10"].astype(float) + 0.5)


TEMP_ORDER = {"freezing": 0, "cold": 1, "cool": 2,
              "mild": 3, "warm": 4, "hot": 5}
df['temp'] = df['temp'].map(TEMP_ORDER).astype(int)

WEATHER_ORDER = {"clear": 0, "cloudy": 1, "bad": 2}
df['weather'] = df['weather'].map(WEATHER_ORDER).astype(int)

df['resp'] = df['response'].map({
    'no_send': 0,
    'no_response': 1,
    'good': 2,
    'bad': 3
}).fillna(3).astype(int)

loc_cats = sorted(set(df["loc"].astype(str)))
loc_map = {c: i for i, c in enumerate(loc_cats)}
df["loc"] = df["loc"].astype(str).map(loc_map).astype(int)

df = df[['uid', 'study_day', 'weekday', 'slot', 'weather', 'temp', 'loc',
         'send', 'dosage', 'resp', 'steps30pre', 'reward']]

df.to_csv('../data/data_eval.csv', index=False)